## Feature Engg.
# Create new features that help predict AQI and make the model smarter.

In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/Rahul Nag/projects/self/durgapur-aqi-analytics/data/processed/aqi_cleaned.csv", parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

In [16]:
# ── Time features ─────────────────────────────────────────────────
df['DayOfWeek'] = df['Date'].dt.dayofweek          # 0=Mon, 6=Sun
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
df['DayOfYear'] = df['Date'].dt.dayofyear

In [17]:
# Season encoding
season_map = {12:0, 1:0, 2:0,   # Winter = 0
              3:1, 4:1, 5:1,     # Spring = 1
              6:2, 7:2, 8:2, 9:2, # Monsoon = 2
              10:3, 11:3}         # Post-Monsoon = 3
df['Season_Code'] = df['Month'].map(season_map)

In [19]:
# ── Lag features (yesterday's and last week's AQI) ────────────────
df['AQI_lag1'] = df['aqi'].shift(1)   # Yesterday's AQI
df['AQI_lag2'] = df['aqi'].shift(2)   # 2 days ago
df['AQI_lag7'] = df['aqi'].shift(7)   # Same day last week

In [20]:
# ── Rolling statistics ────────────────────────────────────────────
df['AQI_roll3_mean'] = df['aqi'].rolling(3).mean()   # 3-day average
df['AQI_roll7_mean'] = df['aqi'].rolling(7).mean()   # 7-day average
df['AQI_roll7_std']  = df['aqi'].rolling(7).std()    # 7-day volatility

In [21]:
# ── Pollutant ratios ──────────────────────────────────────────────
if 'pm25' in df.columns and 'pm10' in df.columns:
    df['PM_ratio'] = df['pm25'] / (df['pm10'] + 1)  # +1 avoids division by zero

# ── Weather interaction features ──────────────────────────────────
if 'windspeed_10m_max' in df.columns:
    df['is_calm_day'] = (df['windspeed_10m_max'] < 5).astype(int)  # Low wind = more pollution

if 'precipitation_sum' in df.columns:
    df['is_rainy_day'] = (df['precipitation_sum'] > 2).astype(int) # Rain washes pollutants

# ── AQI spike flag ────────────────────────────────────────────────
df['is_spike'] = (df['aqi'] > 300).astype(int)  # Binary: Very Poor or worse

In [ ]:
# ── Drop rows with NaN from lag features ──────────────────────────
df_features = df.dropna(subset=['AQI_lag7', 'AQI_roll7_mean']).reset_index(drop=True)
print(f"Final feature set: {df_features.shape}")
print(df_features.columns.tolist())
df_features.to_csv("C:/Users/Rahul Nag/projects/self/durgapur-aqi-analytics/data/processed/aqi_features.csv", index=False)
print("Feature set saved.")

Final feature set: (1454, 31)
['city', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'aqi', 'Date', 'aqi_outlier', 'AQI_Category', 'temperature_2m_max', 'temperature_2m_min', 'windspeed_10m_max', 'precipitation_sum', 'DayOfWeek', 'Month', 'Quarter', 'IsWeekend', 'DayOfYear', 'Season_Code', 'AQI_lag1', 'AQI_lag2', 'AQI_lag7', 'AQI_roll3_mean', 'AQI_roll7_mean', 'AQI_roll7_std', 'PM_ratio', 'is_calm_day', 'is_rainy_day', 'is_spike']
Feature set saved.
